# Open the ASCII exported file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import pandas as pd
from scipy.interpolate import interp1d
%matplotlib widget

def heatmap_interactive(_x, _y, _data, _title, _cmap='jet', _symlog=False, _linthresh=1.0,_loglog=False, _lines = False):
    fig = plt.figure(figsize=(8, 8))
    gs = gridspec.GridSpec(2, 2, width_ratios=[1, 0.5], height_ratios=[0.5, 1], hspace=0.2, wspace=0.2)
    ax_main = plt.subplot(gs[1, 0])
    main_plot = ax_main.pcolormesh(_x, _y, _data, cmap=_cmap)
    ax_main.set(xlabel='Delay / ps', ylabel='Wavelength / nm')
    # set mixed log-lin scale with threshold value linthresh
    if _loglog:
        ax_main.set_xscale('log')
        ax_main.set_yscale('log')
    elif _symlog:
        ax_main.set_xscale('symlog', linthresh=_linthresh)
    # set axis range to min and max values
    #ax_main.set_xlim(_x[0],_x[-1])
    ax_main.set_xlim(-1,_x[-1])
    ax_main.set_ylim(_y[0],_y[-1]) 

    if _lines:
        # Overlay contour lines
        levels = np.linspace(np.min(_data), np.max(_data), 12)
        contour = ax_main.contour(_x, _y, _data, levels=levels, colors='black', linewidths=0.5)
        ax_main.clabel(contour, fmt="%.0e", fontsize=8)

    ax_kin = plt.subplot(gs[0, 0])
    line_kin, = ax_kin.plot(_x,np.zeros(_x.shape))
    kin_zero_line, = ax_kin.plot([_x[0],_x[-1]],[0,0], color="0.6")
    ax_kin.set_xlim(_x[0],_x[-1])
    # Kinetics plot y-axis (intensity)
    if _loglog:
        ax_kin.set_yscale('linear')  # keep linear, unless you want to log this too
    elif _symlog:
        ax_kin.set_yscale('symlog', linthresh=_linthresh)

    ax_spec = plt.subplot(gs[1, 1])
    line_spec, = ax_spec.plot(np.zeros(_y.shape),_y)
    spec_zero_line, = ax_spec.plot([0,0],[_y[0],_y[-1]], color="0.6")
    ax_spec.set_ylim(_y[0],_y[-1])        
    
    # This lower bounds list is necessary because the blocks in the 2D-plot cover a certain range
    def create_lower_bounds(_value_list):
        result = np.empty_like(_value_list)
        #first lower bound is equal to the lowest value in the nm-list
        result[0] = _value_list[0]
        #example: lower bound for 100 ps is 97.5 ps if the value prior is 95 ps, and 75 ps if the value prior is 50 ps.
        for i in range(1,len(_value_list)):
            result[i] = (_value_list[i]+_value_list[i-1])/2
        return result    
    
    nm_lower_bounds = create_lower_bounds(_y)
    time_lower_bounds = create_lower_bounds(_x)
    
    def nm_to_index(_nm):
        return np.where(_nm > nm_lower_bounds)[0][-1]
    
    def time_to_index(_time):
        return np.where(_time > time_lower_bounds)[0][-1]
    
    def mouse_move(event):
        x = event.xdata
        y = event.ydata
        if x is not None and y is not None:
            if x>=_x[0] and x<=_x[-1] and y>=_y[0] and y<=_y[-1]:
                # update spectra slice and rescale
                new_spec = _data[:,time_to_index(x)]
                line_spec.set_xdata(new_spec)
                spec_bounds = ax_spec.get_ylim()
                spec_range = new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()-new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()
                ax_spec.set_xlim(new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()-0.1*spec_range,new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()+0.1*spec_range)            

                # update kinetic slice and rescale
                new_kin = _data[nm_to_index(y),:]
                line_kin.set_ydata(new_kin)
                kin_bounds = ax_kin.get_xlim()  
                kin_range = new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()-new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()                
                ax_kin.set_ylim(new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()-0.1*kin_range,new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()+0.1*kin_range)
                
                # redraw figure
                fig.canvas.draw_idle()
             
    fig.canvas.mpl_connect('motion_notify_event', mouse_move) 
    
    # find max absolute value of 2D data in the specified zoom mode of the plot
    def get_maxvalue(_xlim, _ylim, _xvals, _yvals, _data_array):
        y_filter = (_yvals>=_ylim[0]) & (_yvals<=_ylim[1])
        x_filter = (_xvals>=_xlim[0]) & (_xvals<=_xlim[1])
        
        if not np.all(y_filter == False) and not np.all(x_filter == False):
            return np.amax(np.abs(_data_array[y_filter][:,x_filter]))
        else:
            return 0
    
    def on_xlims_change(event_ax):
        ax_kin.set_xlim(event_ax.get_xlim())
        
        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)

    def on_ylims_change(event_ax):
        ax_spec.set_ylim(event_ax.get_ylim())
        
        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)        

    ax_main.callbacks.connect('xlim_changed', on_xlims_change)
    ax_main.callbacks.connect('ylim_changed', on_ylims_change)
    main_plot.set_clim(vmin=np.min(_data), vmax=np.max(_data))
    fig.colorbar(main_plot, ax=[ax_main, ax_spec], orientation='vertical', label='Intensity (a.u.)')
    plt.show(block=False)




# Open Ldm files

In [ ]:
# Load the .ldm file assuming it's space-separated and has no header
filename = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu _ Mars 25/Ana files/FINAL_Full_spec/Cu(dchtmp)_replicate/2nd_look/dataset_dchtmp_replicate_lda_03_090.dst_a'
data = pd.read_csv(filename, delim_whitespace=True, header=None, names=['lifetime','wavelength', 'intensity'])

# Extract unique coordinates
wavelengths = np.sort(data['wavelength'].unique())
lifetimes = np.sort(data['lifetime'].unique())

# Pivot the data into a 2D grid: rows -> wavelength, columns -> lifetime
intensity_grid = data.pivot(index='wavelength', columns='lifetime', values='intensity').to_numpy()

# Create xarray DataArray
da = xr.DataArray(
    intensity_grid,
    coords={'wavelength': wavelengths, 'lifetime': lifetimes},
    dims=['wavelength', 'lifetime'],
    name='intensity'
)

# Convert to xarray Dataset
ds = xr.Dataset({'intensity': da})

# Display dataset
ds


In [ ]:
heatmap_interactive(ds.lifetime,ds.wavelength, ds['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_loglog=False,_cmap='seismic',_lines = True) #dipp

In [ ]:
upper_time = 0.250
lower_time = 0.05

section = ds.sel(lifetime = slice(lower_time,upper_time))
integral = section.integrate(coord='lifetime')

# Plot result vs wavelength
plt.figure(figsize=(12, 6))
plt.plot(integral['wavelength'], np.abs(integral['intensity']))
plt.xlabel('Wavelength (nm)')
plt.ylabel('Abs Integrated intensity (a.u.)')
plt.title(f'Integrated signal from {lower_time} to {upper_time} ps')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Open 3D data map

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr

# Step 1: Load file
filename = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu _ Mars 25/FINAL_Full_spec/Cu(dchtmp)/2nd look/dataset_dchtmp_lda_00_067.dst_a'
df = pd.read_csv(filename, delim_whitespace=True, header=None, names=['lifetime', 'wavelength', 'intensity'])

trace_length = 187  # Corrected block size
num_traces = df.shape[0] // trace_length

# Extract coordinates
lifetimes = df['lifetime'].values[:trace_length]
wavelengths = df['wavelength'].values[::trace_length]

# Reshape the intensity array
intensity_values = df['intensity'].values.reshape((num_traces, trace_length))

# Create xarray DataArray
da = xr.DataArray(
    data=intensity_values,
    coords={'wavelength': wavelengths, 'lifetime': lifetimes},
    dims=['wavelength', 'lifetime'],
    name='intensity'
)
# Convert to xarray Dataset
dat = xr.Dataset({'intensity': da})



In [ ]:
heatmap_interactive(dat.lifetime, dat.wavelength, dat['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_symlog=False)

In [ ]:
# Define the filename
filename = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu _ Mars 25/FINAL_Full_spec/Cu(dchtmp)/2nd look/dataset_dchtmp_lda_00_067.ana'

# Open the file for writing
with open(filename, 'w') as f:
    # Write the header
    f.write("%FILENAME={}\n".format(filename))
    f.write("%DATATYPE=TAVIS\n")
    f.write("%NUMBERSCANS=1\n")
    f.write("%TIMESCALE=ps\n")

    # Write the time list
    f.write("%TIMELIST={}\n".format(" ".join(f"{value:.2f}" for value in dat.lifetime.values)))
    
    # Write the wavelength list
    f.write("%WAVELENGTHLIST={}\n".format(" ".join(f"{value:.2f}" for value in dat.wavelength.values)))
    
    # Write the intensity matrix
    intensity_matrix = np.transpose(dat.intensity.values)  # Get the intensity values
    i = 0
    for row in intensity_matrix:
        if i == 0:
            f.write("%INTENSITYMATRIX=\n {}\n".format(" ".join(f"{value:.5f}" for value in row)))
            i = i+1
        else:
            f.write("{}\n".format(" ".join(f"{value:.5f}" for value in row)))

print(f"File '{filename}' has been created successfully.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors

plt.figure(figsize=(10, 6))

scaled = dat.intensity


# Use xarray's built-in plot for 2D heatmap
#vmin = -10e-3
vmin = scaled.min().item()
#vmax = 6e-3
vmax = scaled.max().item()

norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

p = scaled.plot(
    x='lifetime', y='wavelength',
    cmap='seismic',  # or viridis, plasma, etc.
    norm=norm,
)



# Increase tick font size
p.colorbar.set_label( 'Intensity (OD)', fontsize=25)
p.axes.tick_params(labelsize=20)

# Optional: set axis titles if needed
p.axes.set_xlim(-0.25, 8)  # Only positive values allowed for log scale
p.axes.set_xlabel("time [ps]", fontsize=25)
p.axes.set_ylabel("Wavelength [nm]", fontsize=25)
#p.axes.set_title("Lifetime Map, fontsize=16)
# Scale colorbar ticks ×10³
def sci_formatter(x, pos):
    return f'{x*1e3:.0f}'  # or use scientific notation: f'{x*1e3:.0e}'

p.colorbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(sci_formatter))
p.colorbar.ax.tick_params(labelsize=20)  # Increase font size of ticks
p.colorbar.set_label(' Intensity [mOD]', fontsize=20)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Split the data manually
linear_region = dat.sel(lifetime=slice(-1, 1))
log_region = dat.sel(lifetime=slice(1.001, 8))  # avoid zero in log

fig, ax = plt.subplots(figsize=(10, 6))

# Plot linear region
p1 = linear_region.intensity.plot(
    ax=ax,
    x='lifetime', y='wavelength',
    cmap='jet',
    add_colorbar=False,
)

# Create second axis for log part
ax_log = ax.twinx()  # dummy axis to share the y-axis (we’ll remove y ticks later)
ax_log.set_position(ax.get_position())  # align exactly

p2 = log_region.intensity.plot(
    ax=ax_log,
    x='lifetime', y='wavelength',
    cmap='jet',
    add_colorbar=True,
    cbar_kwargs={'label': 'Intensity (mOD)'}
)

# Set log x-scale for second axis
ax_log.set_xscale('log')
ax_log.set_xlim(1, 8)

# Format the axes
ax.set_xlim(-1, 1)
ax.set_xlabel("Lifetime (ps)", fontsize=16)
ax.set_ylabel("Wavelength (nm)", fontsize=16)
ax.tick_params(labelsize=14)
ax_log.tick_params(labelsize=14)

# Hide duplicate y-axis
ax_log.get_yaxis().set_visible(False)

plt.tight_layout()
plt.show()


# Import the CA map from Optimus and perform the chirp correction

In [ ]:



# Provided Optimus parameters
lambda_c = 477  # nm
c0 = 0.408     # ps
dispersion_coeffs = [1.791, 0.435,-0.376]  # c1, c2, c3

# Example: your dataset
wavelengths = dat['wavelength'].values
time = dat['lifetime'].values
data = np.transpose(dat['intensity'].values)  # shape (n_time, n_wavelength)

# Step 1: Calculate c(λ) for each wavelength
delta = (wavelengths - lambda_c) / 100
c_lambda = c0 + sum(c * delta**(i + 1) for i, c in enumerate(dispersion_coeffs))  # shape (n_wavelength,)

# Step 2: Interpolate and apply the shift
data_corrected = []

for i, shift in enumerate(c_lambda):
    trace = data[:, i]
    f_interp = interp1d(time - shift, trace, bounds_error=False, fill_value=np.nan)
    trace_shifted = f_interp(time)
    data_corrected.append(trace_shifted)

# Step 3: Reassemble into xarray
data_corrected = np.column_stack(data_corrected)

data_dispersion_corrected = xr.DataArray(
    data_corrected,
    coords={"lifetime": time, "wavelength": wavelengths},
    dims=["lifetime", "wavelength"],
    name="data_dispersion_corrected"
)



In [ ]:

heatmap_interactive(data_dispersion_corrected.lifetime, data_dispersion_corrected.wavelength, data_dispersion_corrected.transpose('wavelength','lifetime'),'Averaged scan plot')

# X ascii file

In [ ]:
# Path to your file
file_path ='/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Co_WK_6psshortcan_lda_01_094.dst_a_c'
# Load the raw data
raw_data = np.loadtxt(file_path)


wavelengths = raw_data[0, 1:] 


time = raw_data[1:,0]     


intensity_2d = raw_data[1:, 1:]  


#Create an xarray to manipulate the data (much easier)
dataset = xr.Dataset(
    {
        "data": (["time","spectral",], intensity_2d)
    },
    coords={
        "time": time,
        "spectral": wavelengths,
    }
)




# Print the dataset
print(dataset)
heatmap_interactive(dataset.time, dataset.spectral, dataset['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

# Average everything in one dimension

In [ ]:
averaged_data = data_dispersion_corrected.mean(dim="wavelength")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(averaged_data.lifetime, averaged_data, label='Averaged Signal')
plt.xlabel('delay (ps)')
plt.ylabel('Intensity (a.u.)')
plt.title('Averaged Signal')
plt.xlim(0.36, 7)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
selected_data = averaged_data.sel(lifetime=slice(0.5,5))
# Define the biexponential function
def bi_exp(t, a1, b1, a2, b2):
    return a1 * np.exp(b1 * t) + a2 * np.exp(b2 * t)

# Data
t = selected_data.lifetime.values
y = selected_data.values

# Initial guess for parameters: [a1, b1, a2, b2]
# a1 and a2 should be positive, b1 and b2 should be negative
p0 = [0.002, -0.1, 0.001, -0.01]

# Bounds: a1, a2 > 0; b1, b2 < 0
bounds = ([0, -np.inf, 0, -np.inf], [np.inf, 0, np.inf, 0])

# Perform the curve fit
popt, pcov = curve_fit(bi_exp, t, y, p0=p0, bounds=bounds)

# Extract the optimal parameters
a1, b1, a2, b2 = popt

# Generate the fitted curve
y_fit = bi_exp(t, a1, b1, a2, b2)

# Detrend the data by subtracting the fitted biexponential
y_detrended = y - y_fit

# Plot the original data, fitted biexponential, and detrended data
plt.figure(figsize=(12, 6))
plt.plot(t, y_detrended)
plt.xlabel('time delay (ps)')
plt.title('Averaged signal')
plt.legend()
plt.show()


In [ ]:
from scipy.fft import fft, fftfreq

selected_data = averaged_data.sel(lifetime=slice(0.5,5))
t = selected_data.lifetime.values
x = y_detrended


dt = t[1] - t[0]  # sampling interval in ps
N = len(t)
freq = fftfreq(N, d=dt)  # in THz

# Convert to wavenumbers (cm⁻¹): 1 THz ≈ 33.356 cm⁻¹
wavenumbers = freq * 33.356
spectrum = np.abs(fft(x)) 


plt.figure(figsize=(8, 5))
plt.plot(wavenumbers, spectrum)
plt.xlabel("Frequency (cm⁻¹)")
plt.ylabel("FFT Amplitude")
plt.xlim(0, 500)
plt.grid('On')
plt.show()

In [ ]:
section = data_dispersion_corrected.sel(lifetime = slice(-0.2,0.2))
integral_val = [] #Initialization of the array containing the min position
for k,wv in enumerate(section.wavelength):
    position_min = section.sel(wavelength = wv).idxmin().item()
    if section.sel(wavelength = wv).min().item() < 0:
        integral_val.append(np.abs(section.sel(wavelength = wv, lifetime = slice(position_min-0.1,position_min+0.05)).integrate(coord='lifetime').item()))
    else: 
        integral_val.append(0)



# Plot result vs wavelength
plt.figure(figsize=(12, 6))
plt.plot(section.wavelength.values, integral_val)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Abs Integrated intensity (a.u.)')
plt.title(f'Integrated early time signal - dpp')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot comparaison before and after interpolation

In [ ]:
target_wavelength = 555

# For `dat` (intensity vs. lifetime)
trace_lifetime = dat.intensity.sel(wavelength=target_wavelength, method='nearest')

trace_After= dataset.data.sel(spectral=target_wavelength, method='nearest')
colors = 'Purple'
plt.figure(figsize=(6, 4))


plt.scatter(dataset.time, trace_After*1000, label=f'{target_wavelength} nm', alpha = 0.5,linewidth=2, color=colors)
plt.plot(dat.lifetime, trace_lifetime*1000, label='Fit', linewidth=2,color=colors)



plt.xlabel('Time [ps]',fontsize=16)
plt.ylabel('Intensity [mOD]',fontsize=16)
#plt.title(f'Kinetic trace at ~{target_wavelength} nm')
plt.axhline(y=0, color='grey', linestyle='-', linewidth=0.5)
plt.xlim(-1,8)
plt.legend(fontsize=14, loc= 'lower right')                # Legend font size
#plt.yticks([-0.0005,0,0.0005,0.001,0.0015,0.002,0.0025,0.003])
plt.tick_params(axis='both', labelsize=14)  # Axis tick font size
plt.grid(True, axis ='x')
plt.xlim(-1,2)
plt.tight_layout()
plt.show()


# X ASCII map

In [ ]:
raw_data[1:, 1:].shape 

In [ ]:
raw_data[0, 1:].shape

In [ ]:
raw_data[1:,0].shape 

In [ ]:
# Path to your file
file_path ='/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu_August25/analysis/Cu(dmp)2_THF_2mmPL_OD0.46_FGS600_MA_SP570_1mw/Cu(dmp)2_gla_03_001_x.dst_c'
# Load the raw data
raw_data = np.loadtxt(file_path)


wavelengths = raw_data[0, 1:] 


time = raw_data[1:,0]     


intensity_2d = raw_data[1:, 1:]  


#Create an xarray to manipulate the data (much easier)
dataset = xr.Dataset(
    {
        "data": (["time","spectral",], intensity_2d)
    },
    coords={
        "time": time,
        "spectral": wavelengths,
    }
)




# Print the dataset
print(dataset)
heatmap_interactive(dataset.time, dataset.spectral, dataset['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

# XY ASCII map

In [ ]:
wavelengths

In [ ]:
time_2d

In [ ]:
import numpy as np
import xarray as xr

# Path to your file
file_path ='/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu_August25/analysis/Cu(dmp)2_THF_2mmPL_OD0.46_FGS600_MA_SP570_1mw/Cu(dmp)2_gla_03_001_x.dst_c'
# Load the raw data
raw_data = np.loadtxt(file_path)

# Extract 1D array of wavelengths from the first row (odd indices)
wavelengths = raw_data[0, 1::2]  # Shape: (241,)

# Extract all time columns (even indices starting from 0), starting from second row
time_2d = raw_data[1:, 0::2]     # Shape: (187, 241)

# Extract all intensity values (odd indices starting from 1), starting from second row
intensity_2d = raw_data[1:, 1::2]  # Shape: (187, 241)

# Check shape consistency
assert time_2d.shape == intensity_2d.shape, "Time and intensity dimensions do not match"

# Create the xarray.Dataset
ds = xr.Dataset(
    data_vars={
        "intensity": (("scan", "wavelength"), intensity_2d),
        "time": (("scan", "wavelength"), time_2d)
    },
    coords={
        "wavelength": ("wavelength", wavelengths),
        "scan": np.arange(intensity_2d.shape[0])
    }
)



# Define a common time axis (e.g., 200 points between min and max)
common_time = np.linspace(ds.time.min().item(), ds.time.max().item(), 200)

# Interpolate intensity onto common time grid per wavelength
interpolated = np.empty((len(common_time), len(ds.wavelength)))

for j, wl in enumerate(ds.wavelength.values):
    # One time trace for this wavelength
    t = ds.time[:, j].values
    y = ds.intensity[:, j].values

    # Interpolate, handle potential issues with duplicate time values
    try:
        f = interp1d(t, y, kind='linear', bounds_error=False, fill_value=np.nan)
        interpolated[:, j] = f(common_time)
    except Exception as e:
        interpolated[:, j] = np.nan


#Create an xarray to manipulate the data (much easier)
dataset = xr.Dataset(
    {
        "data": (["time","spectral",], interpolated)
    },
    coords={
        "time": common_time,
        "spectral": ds.wavelength.values,
    }
)

# Print the dataset
print(dataset)
heatmap_interactive(dataset.time, dataset.spectral, dataset['data'].transpose('spectral','time'),'Averaged scan plot',_symlog=False)

In [ ]:
plt.figure(figsize=(10, 6))
# Multiply intensity by 1000
scaled = dataset.data * 1000

# Plot
p = scaled.plot(
    x='time', y='spectral',
    cmap='jet',
    vmin=-6, vmax=5,
    cbar_kwargs={'label':  'Intensity (mOD)',
                'extend': 'neither'
                    }  # Provide label only
)

# Manually set colorbar label font size
p.colorbar.set_label( 'Intensity (mOD)', fontsize=14)
p.colorbar.ax.tick_params(labelsize=12)  # Colorbar tick font size

# Increase axis font sizes
p.axes.set_xlim(-1,8)
p.axes.set_xlabel("Time (ps)", fontsize=14)
p.axes.set_ylabel("Wavelength (nm)", fontsize=14)
#p.axes.set_title("Kinetic Map", fontsize=16)
p.axes.tick_params(labelsize=12)

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))

scaled = dataset*1000

# Use xarray's built-in plot for 2D heatmap

p = scaled.plot(
    x='time', y='wavelength',
    cmap='jet',  # or viridis, plasma, etc.
    cbar_kwargs={'label': 'Intensity (mOD)'}
)

# Increase tick font size
p.colorbar.set_label( 'Intensity (mOD)', fontsize=16)
p.axes.tick_params(labelsize=14)

# Optional: set axis titles if needed
p.axes.set_xlim(-1,8)
p.axes.set_xlabel("Lifetime (ps)", fontsize=16)
p.axes.set_ylabel("Wavelength (nm)", fontsize=16)
#p.axes.set_title("Lifetime Map", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
dataset.to_netcdf('/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/FINAL_Full_spec/Cu(dipp)/dataset_dipp_lda_01_068.nc')

In [ ]:
# List of wavelengths to plot
selected_wavelengths = [470]  # change these as needed

# Plot
plt.figure(figsize=(8, 6))

for wl in selected_wavelengths:
    # Find the closest available wavelength in the data
    nearest_wl = dataset.spectral.sel(spectral=wl, method="nearest").item()
    
    # Extract the kinetic trace
    trace = dataset.sel(spectral=nearest_wl)
    
    # Plot
    #plt.plot(dataset.time, trace.data.values, label=f"{nearest_wl:.1f} nm") #/np.nanmax(trace.data.values)
    plt.scatter(dataset.time, trace.data.values, label=f"{selected_wavelengths[0]} nm", color = "Blue") #/np.nanmax(trace.data.values)
plt.axhline(y=0, color='grey', linestyle='-', linewidth=2)
plt.xlabel("Time [ps]",fontsize = 20)
plt.ylabel('∆A [OD]',fontsize = 20)
plt.xlim(-1, 2)  # Adjust as needed
#plt.title("Cu Kinetic Traces at Selected Wavelengths")
plt.legend(fontsize = 20, loc= 'upper right')
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Define the filename
filename ='/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/460to520nm/Round_3_optimization/Raw data ANA/dataset_dipp_replicate_lda_02_075.ana'
# Open the file for writing
with open(filename, 'w') as f:
    # Write the header
    f.write("%FILENAME={}\n".format(filename))
    f.write("%DATATYPE=TAVIS\n")
    f.write("%NUMBERSCANS=1\n")
    f.write("%TIMESCALE=ps\n")

    # Write the time list
    f.write("%TIMELIST={}\n".format(" ".join(f"{value:.2f}" for value in dataset.time.values)))
    
    # Write the wavelength list
    f.write("%WAVELENGTHLIST={}\n".format(" ".join(f"{value:.2f}" for value in dataset.spectral.values)))
    
    # Write the intensity matrix
    intensity_matrix = dataset.data.values  # Get the intensity values
    i = 0
    for row in intensity_matrix:
        if i == 0:
            f.write("%INTENSITYMATRIX=\n {}\n".format(" ".join(f"{value:.5f}" for value in row)))
            i = i+1
        else:
            f.write("{}\n".format(" ".join(f"{value:.5f}" for value in row)))

print(f"File '{filename}' has been created successfully.")



In [ ]:

# List of wavelengths to plot
selected_wavelengths = [470,475,478,480,485,490]  # change these as needed

# Plot
plt.figure(figsize=(10, 6))

for wl in selected_wavelengths:
    # Find the closest available wavelength in the data
    nearest_wl = dataset.spectral.sel(spectral=wl, method="nearest").item()
    
    # Extract the kinetic trace
    trace = dataset.sel(spectral=nearest_wl)
    
    # Plot
    plt.plot(dataset.time, trace.data.values/np.nanmax(trace.data.values), label=f"{nearest_wl:.1f} nm")

plt.xlabel("Time (ps)")
plt.ylabel("∆A")
plt.xlim(-1, 1)  # Adjust as needed
plt.title("Kinetic Traces at Selected Wavelengths")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Raw Dataset
# Define the time range of the right wing
right_time_window = dataset.time > 0  # or finer: (dataset.time > 0.1) & (dataset.time < 0.5)

# Select only the right wing
right_wing = dataset.data.sel(time=right_time_window)

# Find minimum (most negative) value over time axis for each wavelength
right_min_amp = right_wing.min(dim="time", skipna=True)

# Plot amplitude vs wavelength
plt.figure(figsize=(8, 5))
plt.plot(dataset.spectral, right_min_amp)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Min ∆A (right wing)")
plt.title("Negative right Wing Amplitude vs Wavelength")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#CA Corrected Dataset
# Define the time range of the right wing
right_time_window = dataset.time > 0  # or finer: (dataset.time > 0.1) & (dataset.time < 0.5)

# Select only the right wing
right_wing = dataset.data.sel(time=right_time_window)

# Find minimum (most negative) value over time axis for each wavelength
right_min_amp = right_wing.min(dim="time", skipna=True)

# Plot amplitude vs wavelength
plt.figure(figsize=(8, 5))
plt.plot(dataset.spectral, right_min_amp)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Min ∆A (right wing)")
plt.title("Negative right Wing Amplitude vs Wavelength for CA corrected data")
plt.grid(True)
plt.tight_layout()
plt.show()


# Open SAS ASCII 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Load Data ===
file_path = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/FINAL_Full_spec/Cu(dmp)_replicate/dataset_dmp_replicate_afterLDA_070_tga_03_001.sas'  # Replace with your actual file path

# Skip the first line and load the rest
data = np.loadtxt(file_path, skiprows=1)

# Extract wavelength and spectra
wavelength = data[:, 0]           # First column = wavelengths
spectra = data[:, 1:]             # Remaining columns = spectra

# === Define labels and colors ===
custom_green = (69/255, 153/255, 82/255)
labels = ['S2', 'S1', 'S1sq', 'T1']
colors = ['blue', custom_green, 'red', 'black']

# === Plotting ===
plt.figure(figsize=(8, 6))

for i in range(min(spectra.shape[1], len(labels))):
    plt.plot(wavelength, spectra[:, i], label=labels[i], color=colors[i],linewidth=3)

plt.xlabel('Wavelength (nm)', fontsize=20)
plt.ylabel('Intensity (a.u.)', fontsize=20)
plt.title('Species associated spectra (SAS)', fontsize=20)
plt.legend(fontsize=14)
plt.ylim(-10e-3,8.5e-3)
plt.axhline(y=0, color='grey', linestyle='-', linewidth=2)
plt.tight_layout()
plt.xticks(fontsize = 18)
plt.yticks(fontsize = 18)
plt.show()


# Final plot


In [ ]:

# Data
complexes = ['Cu(dmp)', 'Cu(dpp)', 'Cu(dipp)', 'Cu(dchtmp)']
avg_lifetimes = [0.105,0.105,0.1,0.085]  # Replace with your actual values
std_devs = [0.005,0.015,0.01,0.005]  # Replace with your actual values
references = [0.045,0.135,2,2]    # literature or reference value
colors = 'blue'



# Plot
plt.figure(figsize=(8, 6))

plt.errorbar(complexes, avg_lifetimes, yerr=std_devs,
             fmt='o', capsize=8, markersize=8, color=colors, linestyle='None')
plt.scatter(complexes,references, marker='^', color=colors, s=100, edgecolor='black', label='Literature Reference')

plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S2 Lifetimes',fontsize=20)
plt.legend(fontsize=14, loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize = 18)
plt.yticks([0.03,0.05,0.07,0.09,0.11,0.13,0.15],fontsize = 18)
plt.ylim(0.01,0.15)
plt.tight_layout()
plt.show()


In [ ]:

# Data
complexes = ['Cu(dmp)', 'Cu(dpp)', 'Cu(dipp)', 'Cu(dchtmp)']
avg_lifetimes = [0.32,0.845,0.79,0.21]  # Replace with your actual values
std_devs = [0.09,0.065,0.07,0.1]  # Replace with your actual values
references = [0.66,0.92,0.6,10]    # literature or reference value
colors = 'green'



# Plot
plt.figure(figsize=(8, 6))

plt.errorbar(complexes, avg_lifetimes, yerr=std_devs,
             fmt='o', capsize=8, markersize=8, color=colors, linestyle='None')
plt.scatter(complexes,references, marker='^', color=colors, s=100, edgecolor='black', label='Literature Reference')

plt.legend(fontsize=14, loc='upper right') 
plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S1sq Lifetimes',fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize = 18)
plt.yticks([0.2,0.4,0.6,0.80],fontsize = 18)
plt.ylim(0,1)
plt.tight_layout()
plt.show()


In [ ]:
# Data
complexes = ['Cu(dmp)', 'Cu(dpp)', 'Cu(dipp)', 'Cu(dchtmp)']
avg_lifetimes = [13.06,11,8.3,5.9]  # Replace with your actual values
std_devs = [1.1,1.8,0.65,0.46]  # Replace with your actual values
references = [7.4,9.4,3.25,4.3]    # literature or reference value
colors = 'red'



# Plot
plt.figure(figsize=(8, 6))

plt.errorbar(complexes, avg_lifetimes, yerr=std_devs,
             fmt='o', capsize=8, markersize=8, color=colors, linestyle='None')
plt.scatter(complexes,references, marker='^', color=colors, s=100, edgecolor='black', label='Literature Reference')

plt.legend(fontsize=14, loc='upper right')  
plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('T1 Lifetimes',fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize = 18)
plt.yticks([2,4,6,8,10,12,14],fontsize = 18)
plt.ylim(2,15)
plt.tight_layout()
plt.show()

In [ ]:


std_devs

In [ ]:
# Data
complexes = ['Cu(dmp)', 'Cu(diptmp)', 'Cu(dchtmp)', 'Cu(dtptmp)','Cu(F-dchtmp)']
avg_lifetimes = [115,120,80,70,120]  # Replace with your actual values
std_devs = [7,14,0,0,0]  # Replace with your actual values
#references = [7.4,9.4,3.25,4.3]    # literature or reference value
colors = 'blue'



# Plot
plt.figure(figsize=(8, 6))

plt.errorbar(complexes, avg_lifetimes, yerr=std_devs,
             fmt='o', capsize=8, markersize=8, color=colors, linestyle='None')
#plt.scatter(complexes,references, marker='^', color=colors, s=100, edgecolor='black', label='Literature Reference')

plt.legend(fontsize=14, loc='upper right')  
plt.ylabel('Lifetime [fs]', fontsize=20)
plt.title('S2 Lifetimes',fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize = 18)
#plt.yticks([2,4,6,8,10,12,14],fontsize = 18)
#plt.ylim(2,15)
plt.tight_layout()
plt.show()

In [ ]:
# Data
complexes = ['Cu(dmp)', 'Cu(diptmp)', 'Cu(dchtmp)', 'Cu(dtptmp)','Cu(F-dchtmp)']
avg_lifetimes = [6,3.86,4.7,4.5,5]  # Replace with your actual values
std_devs = [0.4,1,0.02,1.1,2.1]  # Replace with your actual values
#references = [7.4,9.4,3.25,4.3]    # literature or reference value
colors = 'red'



# Plot
plt.figure(figsize=(8, 6))

plt.errorbar(complexes, avg_lifetimes, yerr=std_devs,
             fmt='o', capsize=8, markersize=8, color=colors, linestyle='None')
#plt.scatter(complexes,references, marker='^', color=colors, s=100, edgecolor='black', label='Literature Reference')

plt.legend(fontsize=14, loc='upper right')  
plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S1sq Lifetimes',fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize = 18)
#plt.yticks([2,4,6,8,10,12,14],fontsize = 18)
#plt.ylim(2,15)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
complexes = ['Cu(dmp)', '_' ,'Cu(dchtmp)',' _' ]
avg_lifetimes = [115,105, 80, 85]
std_devs = [7,5, 0, 5]
colors = ['red', 'blue', 'red', 'blue']  # A list of colors for alternation

# Plot
plt.figure(figsize=(8, 6))

# Loop through the data and apply different colors
for i in range(len(complexes)):
    plt.errorbar(complexes[i], avg_lifetimes[i], yerr=std_devs[i],
                 fmt='o', capsize=8, markersize=8, color=colors[i], linestyle='None')

plt.legend(['new','old'],fontsize=14, loc='upper right')  
plt.ylabel('Lifetime [fs]', fontsize=20)
plt.title('S2 Lifetimes', fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

# Data
complexes = ['Cu(dmp)', '_' ,'Cu(dchtmp)',' _' ]
avg_lifetimes = [6,13.06, 4.7, 5.9]
std_devs = [0.4,1.1, 0.65, 0.46]
colors = ['red', 'blue', 'red', 'blue']  # A list of colors for alternation

# Plot
plt.figure(figsize=(8, 6))

# Loop through the data and apply different colors
for i in range(len(complexes)):
    plt.errorbar(complexes[i], avg_lifetimes[i], yerr=std_devs[i],
                 fmt='o', capsize=8, markersize=8, color=colors[i], linestyle='None')

plt.legend(['new','old'],fontsize=14, loc='upper right')  
plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S1sq Lifetimes', fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Data
complexes = ['Cu(dmp)', 'Cu(dpp)' ,'Cu(dipp)',' Cu(diptmp)','Cu(dchtmp)','Cu(dtptmp)','Cu(F-dchtmp)'] 
avg_lifetimes = [110,150,100,120,85,70,120]
std_devs = [4,15,10,10,5,0,0]
colors = ['red', 'red', 'blue', 'blue','green','green','green']  # A list of colors for alternation

# Plot
plt.figure(figsize=(12, 8))

# Loop through the data and apply different colors
for i in range(len(complexes)):
    plt.errorbar(complexes[i], avg_lifetimes[i], yerr=std_devs[i],
                 fmt='o', capsize=8, markersize=8, color=colors[i], linestyle='None')

# Custom legend handles with specific labels
red_handle = mlines.Line2D([], [], marker='o', color='red', label='1st gen', linestyle='None', markersize=8)
blue_handle = mlines.Line2D([], [], marker='o', color='blue', label='2nd gen', linestyle='None', markersize=8)
green_handle = mlines.Line2D([], [], marker='o', color='green', label='3rd gen', linestyle='None', markersize=8)

# Add the custom legend
plt.legend(handles=[red_handle, blue_handle, green_handle], fontsize=14, loc='upper right')

plt.ylabel('Lifetime [fs]', fontsize=20)
plt.title('S2 Lifetimes', fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Data
complexes = ['Cu(dmp)', 'Cu(dpp)' ,'Cu(dipp)',' Cu(diptmp)','Cu(dchtmp)','Cu(dtptmp)','Cu(F-dchtmp)'] 
avg_lifetimes = [9.5,9.1,8.3,3.9,4.64,4.5,5]
std_devs = [2.1,0.03,0.65,0.75,0.4,0.8,1.5]
colors = ['red', 'red', 'blue', 'blue','green','green','green']  # A list of colors for alternation

# Plot
plt.figure(figsize=(12, 8))

# Loop through the data and apply different colors
for i in range(len(complexes)):
    plt.errorbar(complexes[i], avg_lifetimes[i], yerr=std_devs[i],
                 fmt='o', capsize=8, markersize=8, color=colors[i], linestyle='None')

# Custom legend handles with specific labels
red_handle = mlines.Line2D([], [], marker='o', color='red', label='1st gen', linestyle='None', markersize=8)
blue_handle = mlines.Line2D([], [], marker='o', color='blue', label='2nd gen', linestyle='None', markersize=8)
green_handle = mlines.Line2D([], [], marker='o', color='green', label='3rd gen', linestyle='None', markersize=8)

# Add the custom legend
plt.legend(handles=[red_handle, blue_handle, green_handle], fontsize=14, loc='upper right')

plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S1sq Lifetimes', fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Data
complexes = ['Cu(dmp)', 'Cu(dpp)' ,'Cu(dipp)',' Cu(diptmp)'] 
avg_lifetimes = [0.44,0.85,0.79,0.14]
std_devs = [0.08,0.07,0.07,0.02]
colors = ['red', 'red', 'blue', 'blue']  # A list of colors for alternation

# Plot
plt.figure(figsize=(12, 8))

# Loop through the data and apply different colors
for i in range(len(complexes)):
    plt.errorbar(complexes[i], avg_lifetimes[i], yerr=std_devs[i],
                 fmt='o', capsize=8, markersize=8, color=colors[i], linestyle='None')

# Custom legend handles with specific labels
red_handle = mlines.Line2D([], [], marker='o', color='red', label='1st gen', linestyle='None', markersize=8)
blue_handle = mlines.Line2D([], [], marker='o', color='blue', label='2nd gen', linestyle='None', markersize=8)

# Add the custom legend
plt.legend(handles=[red_handle, blue_handle, green_handle], fontsize=14, loc='upper right')

plt.ylabel('Lifetime [ps]', fontsize=20)
plt.title('S1 Lifetimes', fontsize=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()